In [ ]:
#  미션: “여기어때 숙소 데이터 수집·적재·요약”
# 목표 : 국내 숙소 플랫폼의 공개 웹페이지를 가볍게 탐색해 숙소 메타데이터를 수집하고, 이를 MongoDB에 저장한 뒤 간단한 통계 리포트를 산출
# 범위
# 사이트: https://www.yeogi.com/
# 도시 1곳을 선정(예: 서울)하고, 리스트 페이지 상위 2–3페이지 목표로 수집.
# 수집 항목 (가능한 한 확보 : 자유)
# url(고유), name(숙소명), location(구/동 등 텍스트),
# price_min(최저가, 숫자), rating(평점, 숫자), review_count(리뷰 수, 숫자)
# 아래 데이터를 Aggregation으로 산출.
# 평점 상위 10: rating 내림차순, 동률이면 review_count 많은 순

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import time
from pymongo import MongoClient
import re

service = Service("C:\\Users\\user\\Downloads\\chromedriver\\chromedriver.exe")
options = Options()
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("lang=ko_KR")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36")

driver = webdriver.Chrome(service=service, options=options)

client = MongoClient("mongodb://localhost:27017")
db = client["homework"]
collection = db["jeju_"]

for i in range(3):
    if i == 0:
        url = "https://www.yeogi.com/domestic-accommodations?keyword=%EC%A0%9C%EC%A3%BC%EB%8F%84&checkIn=2025-12-16&checkOut=2025-12-17&personal=2&freeForm=false"
    else:
        url = f"https://www.yeogi.com/domestic-accommodations?keyword=%EC%A0%9C%EC%A3%BC%EB%8F%84&checkIn=2025-12-16&checkOut=2025-12-17&personal=2&freeForm=false&page={i+1}"
    driver.get(url)
    time.sleep(2)
    elements = driver.find_elements(By.CSS_SELECTOR, "a.gc-thumbnail-type-seller-card.css-wels0m")

    for element in elements:
        try:
            name = element.find_element(By.CSS_SELECTOR, "h3.gc-thumbnail-type-seller-card-title.css-1gxx2ac").text
            link = element.get_attribute("href")
            location = element.find_element(By.CSS_SELECTOR, "span.css-1rzfout").text
            price_min = element.find_element(By.CSS_SELECTOR, "span.css-5r5920").text
            rating = element.find_element(By.CSS_SELECTOR, "span.css-9ml4lz").text
            review_count = element.find_element(By.CSS_SELECTOR, "span.css-oj6onp").text.replace("명 평가", "")
        except:
            continue

        doc = {
            "name": name,
            "url": link,
            "location": location,
            "price_min": int(re.sub(r"[^\d]", "", price_min)) if price_min else None,
            "rating": float(rating) if rating else None,
            "review_count": int(re.sub(r"[^\d]", "", review_count)) if review_count else None
        }
        collection.update_one({"url": doc["url"]}, {"$set": doc}, upsert=True)
        print(doc)

time.sleep(2)
driver.quit()


SessionNotCreatedException: Message: session not created: This version of ChromeDriver only supports Chrome version 138
Current browser version is 140.0.7339.82 with binary path C:\Program Files\Google\Chrome\Application\chrome.exe; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
	GetHandleVerifier [0x0x7ff6be73e935+77845]
	GetHandleVerifier [0x0x7ff6be73e990+77936]
	(No symbol) [0x0x7ff6be4f9cda]
	(No symbol) [0x0x7ff6be53e53f]
	(No symbol) [0x0x7ff6be53d58b]
	(No symbol) [0x0x7ff6be536d7d]
	(No symbol) [0x0x7ff6be5329e5]
	(No symbol) [0x0x7ff6be5862ce]
	(No symbol) [0x0x7ff6be585a60]
	(No symbol) [0x0x7ff6be5786a3]
	(No symbol) [0x0x7ff6be541791]
	(No symbol) [0x0x7ff6be542523]
	GetHandleVerifier [0x0x7ff6bea1684d+3059501]
	GetHandleVerifier [0x0x7ff6bea10c0d+3035885]
	GetHandleVerifier [0x0x7ff6bea30400+3164896]
	GetHandleVerifier [0x0x7ff6be758c3e+185118]
	GetHandleVerifier

In [4]:
# 평점 상위 10: rating 내림차순, 동률이면 review_count 많은 순
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017")
db = client["homework"]
collection = db["jeju_"]

pipeline = [
    {"$sort":{"rating" :-1, "review_count" :-1}},
    {"$limit" : 10}
]
jeju_ = db.jeju_
hotels = jeju_.aggregate(pipeline) 
for hotel in hotels :
    print(hotel)

{'_id': ObjectId('68c28f1525a8fcacd48bffde'), 'url': 'https://www.yeogi.com/domestic-accommodations/9888?checkIn=2025-12-16&checkOut=2025-12-17&personal=2', 'location': '서귀포시', 'name': '히든 클리프 호텔 & 네이쳐', 'price_min': 173642, 'rating': 9.5, 'review_count': 2080}
{'_id': ObjectId('68c28f1625a8fcacd48bffe5'), 'url': 'https://www.yeogi.com/domestic-accommodations/6654?checkIn=2025-12-16&checkOut=2025-12-17&personal=2', 'location': '서귀포시', 'name': '해비치 호텔&리조트', 'price_min': 252648, 'rating': 9.5, 'review_count': 880}
{'_id': ObjectId('68c28f1525a8fcacd48bffd6'), 'url': 'https://www.yeogi.com/domestic-accommodations/48099?checkIn=2025-12-16&checkOut=2025-12-17&personal=2', 'location': '서귀포시', 'name': '랜딩관 제주신화월드 호텔앤리조트', 'price_min': 138600, 'rating': 9.4, 'review_count': 2687}
{'_id': ObjectId('68c28f1525a8fcacd48bffd7'), 'url': 'https://www.yeogi.com/domestic-accommodations/54798?checkIn=2025-12-16&checkOut=2025-12-17&personal=2', 'location': '서귀포시', 'name': '신화관 제주신화월드 호텔앤리조트', 'price_min